# Week 17: Capstone Synthesis, Parallel Scan & Randomized Methods

**Z5007 Programming and Data Structures (MTech DSAI Core) · IIT Madras Zanzibar**

Blelloch prefix scan, Monte Carlo estimation, and Zanzibar dispatch synthesis.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5007/week-17/labs/W17_Final_Capstone_Synthesis.ipynb)

Run cells top to bottom (`Runtime > Run all`). Every algorithm is implemented from first principles in Python and verified with assertions.

## 1. Parallel Prefix Sum (Blelloch Scan) & Monte Carlo Estimation

Parallel prefix scan computes all running totals in $O(\log n)$ parallel depth and $O(n)$ total work; Monte Carlo sampling trades deterministic certainty for $O(1/\sqrt{N})$ error scaling.

In [1]:
import heapq
import math
import random


def parallel_prefix_sum(arr):
    n = 1
    while n < len(arr):
        n *= 2
    buf = arr[:] + [0] * (n - len(arr))
    step = 1
    while step < n:
        for i in range(0, n, 2 * step):
            buf[i + 2 * step - 1] += buf[i + step - 1]
        step *= 2
    total = buf[n - 1]
    buf[n - 1] = 0
    step //= 2
    while step >= 1:
        for i in range(0, n, 2 * step):
            t = buf[i + step - 1]
            buf[i + step - 1] = buf[i + 2 * step - 1]
            buf[i + 2 * step - 1] += t
        step //= 2
    return [buf[i + 1] for i in range(len(arr) - 1)] + [total]


arr = [3, 1, 4, 1, 5, 9, 2]
ps = parallel_prefix_sum(arr)
assert ps == [3, 4, 8, 9, 14, 23, 25]
print("Parallel prefix sum:", ps)


Parallel prefix sum: [3, 4, 8, 9, 14, 23, 25]


## 2. Zanzibar Logistics Capstone Synthesis (Dijkstra + 0/1 Knapsack)

Combining graph shortest paths with dynamic-programming cargo optimization solves end-to-end dispatch decisions cleanly.

In [2]:
roads = {"StoneTown": [("Mbweni", 6.0), ("Fumba", 18.0)], "Mbweni": [("Bweleo", 11.0)], "Fumba": [("Bweleo", 5.0)], "Bweleo": []}
cargo = [("Cloves", 180, 20), ("Seedlings", 120, 15), ("Sensors", 210, 25)]
cap = 40

dist = {u: math.inf for u in roads}
dist["StoneTown"] = 0.0
pq = [(0.0, "StoneTown")]
while pq:
    d, u = heapq.heappop(pq)
    if d <= dist[u]:
        for v, w in roads[u]:
            if d + w < dist[v]:
                dist[v] = d + w
                heapq.heappush(pq, (dist[v], v))

dp = [[0] * (cap + 1) for _ in range(len(cargo) + 1)]
for i, (_, val, wt) in enumerate(cargo, 1):
    for w in range(cap + 1):
        dp[i][w] = max(dp[i - 1][w], dp[i - 1][w - wt] + val if wt <= w else 0)

print(f"Shortest StoneTown -> Bweleo = {dist['Bweleo']:.0f} km | Max cargo value (40 kg) = {dp[len(cargo)][cap]}")
assert dist["Bweleo"] == 17.0 and dp[len(cargo)][cap] == 330


Shortest StoneTown -> Bweleo = 17 km | Max cargo value (40 kg) = 330
